# Nextflow Pipeline Resource Analysis

Analyze Nextflow trace files to compare pipeline runs, inspect derived
resource metrics, and highlight processes that are most promising to tune.
## Workflow
1. **Setup** — Import plotting and helper utilities used throughout the notebook
2. **Load & Preprocess** — Read trace files, derive run/process metrics, and flag consistent processes
3. **Inspect Preprocessing Outputs** — Preview the run-level, per-process outputs produced during preprocessing
4. **Interactive Analysis Views** — Explore three complementary dashboards for ranking optimization priorities, comparing per-process resource burdens, and tracking how consistent processes scale across runs


### 1. Setup

Import the plotting libraries and project helpers, then configure a shared
visual style for the interactive charts below.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from ipywidgets import Dropdown

# Add project root to path
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "workflow_helpers.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from workflow_helpers import (
    load_comparison_groups,
    create_process_composition_plot,
    create_optimization_priority_plot,
    create_process_scaling_explorer,
)

# Configure matplotlib styling
plt.style.use("seaborn-v0_8-dark")
plt.rcParams.update(
    {
        "font.family": "sans-serif",
        "font.size": 10,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.15,
        "grid.linestyle": "-",
        "axes.titleweight": "medium",
        "axes.labelweight": "regular",
        "figure.dpi": 120,
    }
)

## 2. Load & Preprocess Trace Files

Define comparison groups, load each trace file, and build both run-level and
per-process summaries. During this step the loader also filters to completed
or cached tasks, computes derived resource metrics, and marks whether each
process is present in every run of its group.

In [2]:
# ── Configuration ──────────────────────────────────────────────────────────────
# Define comparison groups of trace files
comparison_groups = {
    "rnaseq": {
        "trace_files": [
            Path("../data/rnaseq_runs/rnaseq_test_trace_2026-03-19_16-41-59.txt"),
            Path("../data/rnaseq_runs/rnaseq_test_full_trace_2026-03-19_16-51-08.txt"),
        ],
        "labels": ["rnaseq test", "rnaseq test full"],  # or None for auto-labels
    },
}

# ── Load all trace files ───────────────────────────────────────────────────────
run_summary_df, process_df, trace_files, labels = load_comparison_groups(
    comparison_groups
)

Comparison Groups
  Total: 1
  - rnaseq: 2 run(s)

Runs: rnaseq
  - rnaseq test | 219 task(s) | 62 process(es) | rnaseq_test_trace_2026-03-19_16-41-59.txt
  - rnaseq test full | 303 task(s) | 51 process(es) | rnaseq_test_full_trace_2026-03-19_16-51-08.txt
⚠️  WARNING: 27 tasks used more CPU cores than requested!
   Scheduler may not be enforcing CPU limits properly.
   Examples (showing up to 5):
     NFCORE_RNASEQ:RNASEQ:BEDTOOLS_GENOMECOV_REV: requested 1.0 cores, used 1.0 (102.3%)
     NFCORE_RNASEQ:RNASEQ:BEDTOOLS_GENOMECOV_FW: requested 1.0 cores, used 1.0 (102.3%)
     NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_STATS: requested 1.0 cores, used 1.3 (130.5%)
     NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_STATS: requested 1.0 cores, used 1.2 (124.9%)
     NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_STATS: requested 1.0 cores, used 1.2 (122.2%)


Consistency: rna

### Data Overview

These previews show the main tabular outputs of preprocessing:
`run_summary_df` for one row per run and `process_df` for one row per
process within each run, including the `is_consistent` flag computed
during loading. They are useful for checking the derived columns before
moving on to the interactive plots.

In [3]:
import pandas as pd

print("Run summary columns:")
print(run_summary_df.columns.tolist())
print("Example run summary data:")
display(run_summary_df.head(10))


print("Process summary columns:")
print(process_df.columns.tolist())
print("Example process summary data:")
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
display(process_df.sample(n=min(5, len(process_df)), random_state=42))

Run summary columns:
['task_count', 'process_count', 'total_runtime_h', 'cpu_h', 'memory_h', 'io_gb', 'total_io_bytes', 'filename', 'group', 'label']
Example run summary data:


,task_count,process_count,total_runtime_h,cpu_h,memory_h,io_gb,total_io_bytes,filename,group,label
0,219,62,0.181596,0.258934,0.277300,6.173546,6628794368,rnaseq_test_trace_2026-03-19_16-41-59,rnaseq,rnaseq test
1,303,51,44.556156,122.843415,532.128662,4285.407391,4601421148160,rnaseq_test_full_trace_2026-03-19_16-51-08,rnaseq,rnaseq test full


Process summary columns:
['process', 'n_tasks', 'total_runtime_hours', 'cpu_h', 'memory_h', 'io_gb', 'cpu_requested_h', 'memory_requested_h', 'time_requested_h', 'cpu_efficiency', 'mem_efficiency', 'time_efficiency', 'avg_cpus_used', 'std_cpus_used', 'avg_mem_used_gb', 'std_mem_used_gb', 'avg_runtime_hours', 'std_runtime_hours', 'filename', 'group', 'label', 'process_display', 'process_name', 'is_consistent']
Example process summary data:


,process,n_tasks,total_runtime_hours,cpu_h,memory_h,io_gb,cpu_requested_h,memory_requested_h,time_requested_h,cpu_efficiency,mem_efficiency,time_efficiency,avg_cpus_used,std_cpus_used,avg_mem_used_gb,std_mem_used_gb,avg_runtime_hours,std_runtime_hours,filename,group,label,process_display,process_name,is_consistent
80,NFCORE_RNASEQ:RNASEQ:BAM_RSEQC:RSEQC_JUNCTIONANNOTATION,8,1.092960,0.940146,1.631880e-01,77.674442,6.557762,39.346570,64.0,14.336383,0.414745,1.707750,0.862250,0.107641,0.146474,0.025653,0.136620,0.033267,rnaseq_test_full_trace_2026-03-19_16-51-08,rnaseq,rnaseq test full,...UNCTIONANNOTATION,RSEQC_JUNCTIONANNOTATION,True
4,NFCORE_RNASEQ:PREPARE_GENOME:GUNZIP_ADDITIONAL_FASTA,1,0.000037,0.000009,1.293182e-07,0.000755,0.000037,0.000225,4.0,24.900000,0.057475,0.000937,0.249000,0.000000,0.003448,0.000000,0.000037,0.000000,rnaseq_test_trace_2026-03-19_16-41-59,rnaseq,rnaseq test,..._ADDITIONAL_FASTA,GUNZIP_ADDITIONAL_FASTA,False
40,NFCORE_RNASEQ:RNASEQ:FASTQ_QC_TRIM_FILTER_SETSTRANDEDNESS:FASTQ_FASTQC_UMITOOLS_TRIMGALORE:TRIMGALORE,5,0.014444,0.017434,3.798234e-03,0.480095,0.173333,1.040000,80.0,10.057853,0.365215,0.018056,1.174600,0.241488,0.261760,0.047714,0.002889,0.000541,rnaseq_test_trace_2026-03-19_16-41-59,rnaseq,rnaseq test,TRIMGALORE,TRIMGALORE,True
69,NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:SAMTOOLS_INDEX,8,0.196111,0.214947,9.668647e-03,92.881889,0.392222,2.353333,32.0,54.802479,0.410849,0.612847,1.391875,0.506664,0.046901,0.008635,0.024514,0.017232,rnaseq_test_full_trace_2026-03-19_16-51-08,rnaseq,rnaseq test full,...LS:SAMTOOLS_INDEX,BAM_SORT_STATS_SAMTOOLS:SAMTOOLS_INDEX,True
10,NFCORE_RNASEQ:RNASEQ:ALIGN_STAR:BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_IDXSTATS,5,0.000000,0.000000,0.000000e+00,0.052711,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,rnaseq_test_trace_2026-03-19_16-41-59,rnaseq,rnaseq test,...SAMTOOLS_IDXSTATS,BAM_SORT_STATS_SAMTOOLS:BAM_STATS_SAMTOOLS:SAMTOOLS_IDXSTATS,True


### Metric Meanings

- `task_count`: number of completed or cached tasks kept after preprocessing.
- `process_count`: number of unique Nextflow processes in a run.
- `total_runtime_h`: sum of task runtimes in hours.
- `cpu_h`: total CPU core-hours, computed as `sum((%cpu / 100) * runtime_hours)`.
- `memory_h`: peak-memory-weighted GB-hours, computed as `sum(peak_rss_gb * runtime_hours)`. This uses peak RSS, so it is a burden proxy rather than true average memory over time.
- `io_gb`: total I/O transfer volume in gigabytes, computed from `read_bytes + write_bytes` when available, otherwise from `rchar + wchar`.
- `cpu_efficiency`, `mem_efficiency`, `time_efficiency`: used-versus-requested efficiency percentages aggregated per process.
- `avg_*` and `std_*`: per-process averages and spreads across tasks, used later to estimate variability.
- `is_consistent`: `True` when a process appears in every run of its comparison group, which makes it eligible for cross-run scaling views.


## 4. Interactive Analysis Views

This section brings together the three interactive plotting views used to
interpret the processed trace data. Start by ranking promising tuning targets,
then compare resource burdens within a run, and finally inspect how shared
processes change across runs.

### 4.1 Optimization Priority Dashboard

This interactive chart ranks processes by a composite priority score:
`impact × (inefficiency + variability)`.

Use the sliders to rebalance how much CPU, memory, runtime cost,
inefficiency, and variability contribute to the score. The hover details
break the final ranking into its underlying components for the selected run.


In [4]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# -----------------------------
# Controls
# -----------------------------
run_label_widget = widgets.Dropdown(
    options=labels,
    description="Run:",
    style={"description_width": "initial"},
)

top_n_widget = widgets.IntSlider(
    value=20,
    min=5,
    max=30,
    step=1,
    description="Top N:",
    style={"description_width": "initial"},
    continuous_update=False,
)

# Impact sliders
impact_cpu_widget = widgets.FloatSlider(
    value=0.4,
    min=0.0,
    max=1.0,
    step=0.05,
    description="CPU cost",
    style={"description_width": "initial"},
    continuous_update=False,
)
impact_mem_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Memory cost",
    style={"description_width": "initial"},
    continuous_update=False,
)
impact_time_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Time cost",
    style={"description_width": "initial"},
    continuous_update=False,
)

# Inefficiency sliders
ineff_cpu_widget = widgets.FloatSlider(
    value=0.4,
    min=0.0,
    max=1.0,
    step=0.05,
    description="CPU ineff.",
    style={"description_width": "initial"},
    continuous_update=False,
)
ineff_mem_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Memory ineff.",
    style={"description_width": "initial"},
    continuous_update=False,
)
ineff_time_widget = widgets.FloatSlider(
    value=0.3,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Time ineff.",
    style={"description_width": "initial"},
    continuous_update=False,
)

# Variability sliders
var_runtime_widget = widgets.FloatSlider(
    value=0.6,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Runtime CV",
    style={"description_width": "initial"},
    continuous_update=False,
)
var_cpu_widget = widgets.FloatSlider(
    value=0.2,
    min=0.0,
    max=1.0,
    step=0.05,
    description="CPU CV",
    style={"description_width": "initial"},
    continuous_update=False,
)
var_mem_widget = widgets.FloatSlider(
    value=0.2,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Memory CV",
    style={"description_width": "initial"},
    continuous_update=False,
)

# -----------------------------
# Plot function
# -----------------------------
output = widgets.Output()


def show_optimization_dashboard(
    run_label,
    top_n,
    impact_cpu,
    impact_mem,
    impact_time,
    ineff_cpu,
    ineff_mem,
    ineff_time,
    var_runtime,
    var_cpu,
    var_mem,
):
    idx = labels.index(run_label)
    stem = trace_files[idx].stem
    metrics = process_df[process_df["filename"] == stem].copy()

    impact_weights = {
        "cpu_cost": impact_cpu,
        "mem_cost": impact_mem,
        "time_cost": impact_time,
    }
    inefficiency_weights = {
        "cpu_ineff": ineff_cpu,
        "mem_ineff": ineff_mem,
        "time_ineff": ineff_time,
    }
    variability_weights = {
        "runtime_cv": var_runtime,
        "cpu_cv": var_cpu,
        "mem_cv": var_mem,
    }

    fig = create_optimization_priority_plot(
        metrics=metrics,
        impact_weights=impact_weights,
        inefficiency_weights=inefficiency_weights,
        variability_weights=variability_weights,
        top_n=top_n,
    )

    with output:
        clear_output(wait=True)
        fig.show()


# -----------------------------
# Grouped layout in two rows
# -----------------------------
general_box = widgets.VBox(
    [
        widgets.HTML("<h4 style='margin:0;'>General</h4>"),
        widgets.HTML(
            "<div style='font-size:12px;color:#666;'>"
            "Weights are normalized within each section."
            "</div>"
        ),
        run_label_widget,
        top_n_widget,
    ],
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px 12px",
        margin="6px 0 10px 0",
        width="100%",
    ),
)

shared_box_layout = widgets.Layout(
    border="1px solid #ddd",
    padding="10px 12px",
    margin="0 8px 0 0",
    width="33%",
)

impact_box = widgets.VBox(
    [
        widgets.HTML(
            "<h4 style='margin:0;'>Impact</h4>"
            "<div style='font-size:12px;color:#666;'>"
            "Relative weighting of compute cost components"
            "</div>"
        ),
        impact_cpu_widget,
        impact_mem_widget,
        impact_time_widget,
    ],
    layout=shared_box_layout,
)

ineff_box = widgets.VBox(
    [
        widgets.HTML(
            "<h4 style='margin:0;'>Inefficiency</h4>"
            "<div style='font-size:12px;color:#666;'>"
            "Relative weighting of resource inefficiencies"
            "</div>"
        ),
        ineff_cpu_widget,
        ineff_mem_widget,
        ineff_time_widget,
    ],
    layout=shared_box_layout,
)

var_box = widgets.VBox(
    [
        widgets.HTML(
            "<h4 style='margin:0;'>Variability</h4>"
            "<div style='font-size:12px;color:#666;'>"
            "Relative weighting of input-dependent variability"
            "</div>"
        ),
        var_runtime_widget,
        var_cpu_widget,
        var_mem_widget,
    ],
    layout=widgets.Layout(
        border="1px solid #ddd",
        padding="10px 12px",
        margin="0",
        width="33%",
    ),
)

slider_row = widgets.HBox(
    [impact_box, ineff_box, var_box],
    layout=widgets.Layout(
        width="100%",
        align_items="flex-start",
        justify_content="space-between",
    ),
)

controls = widgets.VBox(
    [
        widgets.HTML(
            "<h3 style='margin:0 0 8px 0;'>Optimization Priority Dashboard</h3>"
        ),
        general_box,
        slider_row,
    ]
)

# -----------------------------
# Bind widgets to function
# -----------------------------
interactive = widgets.interactive_output(
    show_optimization_dashboard,
    {
        "run_label": run_label_widget,
        "top_n": top_n_widget,
        "impact_cpu": impact_cpu_widget,
        "impact_mem": impact_mem_widget,
        "impact_time": impact_time_widget,
        "ineff_cpu": ineff_cpu_widget,
        "ineff_mem": ineff_mem_widget,
        "ineff_time": ineff_time_widget,
        "var_runtime": var_runtime_widget,
        "var_cpu": var_cpu_widget,
        "var_mem": var_mem_widget,
    },
)

display(controls, output, interactive)

Output()

Output()

### 4.2 Resource Burden Views

Use this view to compare the dominant CPU, memory, runtime, and I/O burdens
within a single run. Switching the sort metric helps surface which processes
matter most for each resource dimension.


In [5]:
composition_run_widget = Dropdown(options=labels, description="Run:")
composition_sort_widget = Dropdown(
    options=[
        ("CPU", "cpu_h"),
        ("Memory", "memory_h"),
        ("Runtime", "total_runtime_hours"),
        ("I/O", "io_gb"),
    ],
    description="Sort by:",
)
composition_output = widgets.Output()


def show_process_composition_dashboard(run_label, sort_by):
    idx = labels.index(run_label)
    stem = trace_files[idx].stem

    pp = process_df[process_df["filename"] == stem].copy()

    fig = create_process_composition_plot(pp, run_label, sort_by)
    fig.update_layout(width=1100, height=400)

    composition_output.clear_output(wait=True)
    with composition_output:
        fig.show(renderer="plotly_mimetype")


_ = widgets.interactive_output(
    show_process_composition_dashboard,
    {
        "run_label": composition_run_widget,
        "sort_by": composition_sort_widget,
    },
)

display(
    widgets.VBox(
        [
            widgets.HBox([composition_run_widget, composition_sort_widget]),
            composition_output,
        ]
    )
)

### 4.3 Scaling Explorer

This cross-run view focuses on consistent processes only, so the same logical
steps can be compared fairly across runs. It is most useful for spotting which
shared processes grow disproportionately as runtime and total work increase.


In [6]:
TOP_PROCESSES_PER_RESOURCE = 15


def show_scaling_explorer(group_name):
    fig = create_process_scaling_explorer(
        process_df,
        run_summary_df,
        group_name,
        top_n=TOP_PROCESSES_PER_RESOURCE,
    )
    if fig is None:
        print(f"No consistent processes found for {group_name}")
        return
    return fig


if len(comparison_groups) > 1:
    scaling_group_widget = Dropdown(
        options=comparison_groups.keys(),
        description="Group:",
    )
    scaling_output = widgets.Output()

    def render_scaling_explorer(group_name):
        fig = show_scaling_explorer(group_name)
        scaling_output.clear_output(wait=True)
        with scaling_output:
            if fig is not None:
                fig.show(renderer="plotly_mimetype")

    _ = widgets.interactive_output(
        render_scaling_explorer,
        {"group_name": scaling_group_widget},
    )
    display(widgets.VBox([scaling_group_widget, scaling_output]))
else:
    fig = show_scaling_explorer(list(comparison_groups.keys())[0])
    if fig is not None:
        fig.show(renderer="plotly_mimetype")